# Library

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import polars as pl
from pathlib import Path

## Datasets metadata

In [9]:
DATA_DIR = Path("../datathon-2026-round-1")
METADATA = {}

for csv_file in DATA_DIR.glob("*.csv"):
    # Dùng polars đọc nhanh sample
    sample = pl.scan_csv(csv_file).fetch(1000)
    dtypes = {col: str(sample[col].dtype) for col in sample.columns}
    METADATA[csv_file.stem] = {
        "columns": list(sample.columns),
        "dtypes": dtypes,
        "size_mb": csv_file.stat().st_size / 1e6,
    }

    print("="*20 + f" {csv_file.stem} " + "="*20 + "\n")
    print("Columns: ", METADATA[csv_file.stem]['columns'])
    print("Data types: ", METADATA[csv_file.stem]['dtypes'])
    print("size_mb: ", METADATA[csv_file.stem]['size_mb'])


    print("\n\n" + "="*80 + "\n")


==================== web_traffic ====================

Columns:  ['date', 'sessions', 'unique_visitors', 'page_views', 'bounce_rate', 'avg_session_duration_sec', 'traffic_source']
Data types:  {'date': 'String', 'sessions': 'Int64', 'unique_visitors': 'Int64', 'page_views': 'Int64', 'bounce_rate': 'Float64', 'avg_session_duration_sec': 'Float64', 'traffic_source': 'String'}
size_mb:  0.208722



==================== customers ====================

Columns:  ['customer_id', 'zip', 'city', 'signup_date', 'gender', 'age_group', 'acquisition_channel']
Data types:  {'customer_id': 'Int64', 'zip': 'Int64', 'city': 'String', 'signup_date': 'String', 'gender': 'String', 'age_group': 'String', 'acquisition_channel': 'String'}
size_mb:  7.079679



==================== products ====================

Columns:  ['product_id', 'product_name', 'category', 'segment', 'size', 'color', 'price', 'cogs']
Data types:  {'product_id': 'Int64', 'product_name': 'String', 'category': 'String', 'segment': 'Stri

/var/folders/ql/s803k9yn2l570nz7shf781y80000gn/T/ipykernel_91331/3260327731.py:6: DeprecationWarning: `LazyFrame.fetch` is deprecated; use `LazyFrame.collect` instead, in conjunction with a call to `head`.
  sample = pl.scan_csv(csv_file).fetch(1000)


## Mô tả vận hành của dòng dữ liệu

+ Hành trình khách hàng: Từ lúc họ vào web **(web_traffic)**, trở thành khách hàng **(customers)**, thực hiện đơn hàng **(orders)** đến khi để lại đánh giá **(reviews)** hoặc yêu cầu trả hàng **(returns)**.

+ Vận hành nội bộ: Sự kết nối giữa kho bãi **(inventory)**, giao hàng **(shipments)** và các chương trình khuyến mãi phức tạp **(promotions)**.

+ Tài chính & Lợi nhuận: Dòng tiền từ thanh toán **(payments)** đối soát với chi phí hàng bán **(sales, products.cogs)** để tìm ra lợi nhuận thực tế

# Phase 1: Ingestion & Data Quality

In [10]:
import polars as pl

customer_journey_tables = ["web_traffic", "customers", "reviews", "orders", "order_items", "returns"]
internal_operation_tables = ["inventory", "shipments", "promotions", "geography"]
financial_profit_tables = ["payments", "sales", "products"]

# 1. LOAD RAW DATA
def load_tables(table_list):
    return {
        name: pl.read_csv(f"{DATA_DIR}/{name}.csv")
        for name in table_list
    }

raw_dfs = {
    "customer_journey": load_tables(customer_journey_tables),
    "internal_operation": load_tables(internal_operation_tables),
    "financial_profit": load_tables(financial_profit_tables),
}

# 2. BASIC PROFILING FUNCTION
def profile_df(name, df):
    print(f"\n=== {name} ===")
    print("=== Data Statistic ===")
    print(df.describe())

    print("\n=== Null Count ===")
    print(df.null_count())

    print("\n=== Duplicate Rows ===")
    print(df.select(pl.all().is_duplicated().sum()))

for group_name, tables in raw_dfs.items():
    print(f"\n\n========= {group_name.upper()} =========")
    
    for table_name, df in tables.items():
        profile_df(table_name, df)



========= CUSTOMER_JOURNEY =========

=== web_traffic ===
=== Data Statistic ===
shape: (9, 8)
┌────────────┬────────────┬────────────┬───────────┬───────────┬───────────┬───────────┬───────────┐
│ statistic  ┆ date       ┆ sessions   ┆ unique_vi ┆ page_view ┆ bounce_ra ┆ avg_sessi ┆ traffic_s │
│ ---        ┆ ---        ┆ ---        ┆ sitors    ┆ s         ┆ te        ┆ on_durati ┆ ource     │
│ str        ┆ str        ┆ f64        ┆ ---       ┆ ---       ┆ ---       ┆ on_sec    ┆ ---       │
│            ┆            ┆            ┆ f64       ┆ f64       ┆ f64       ┆ ---       ┆ str       │
│            ┆            ┆            ┆           ┆           ┆           ┆ f64       ┆           │
╞════════════╪════════════╪════════════╪═══════════╪═══════════╪═══════════╪═══════════╪═══════════╡
│ count      ┆ 3652       ┆ 3652.0     ┆ 3652.0    ┆ 3652.0    ┆ 3652.0    ┆ 3652.0    ┆ 3652      │
│ null_count ┆ 0          ┆ 0.0        ┆ 0.0       ┆ 0.0       ┆ 0.0       ┆ 0.0       ┆ 0     